# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OjaswiGautam/FlyrankAI/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Per the training-honest-models skill file's own method-selection table — "grouping items → K-Means (pick k with silhouette), then NAME clusters after inspecting them → unsupervised needs human naming" — K-Means is the directly recommended method for this exact question shape, and it matches Lane 3's task framing established back in w02: this is unsupervised archetype discovery, not classification or ranking, so there is no label to fit against and no precision@K to chase.

Why K-Means specifically, over the toolkit's other options:

*   Not Logistic Regression / Random Forest / Gradient Boosting — these require a supervised label. Lane 3 has none, by design (w02 established this explicitly: inventing a proxy label just to use a supervised method would mean fabricating the exact "ground truth" this lane is supposed to discover, not assume).

*   Not correlation/signal analysis alone — useful as a diagnostic (and we used it in w04's signal checks), but it answers "does X relate to Y," not "what groups exist." It doesn't produce archetypes.

*   K-Means over other clustering methods (e.g., GMM, HDBSCAN) — K-Means is the method the skill file names directly for this task shape, it produces hard, interpretable cluster assignments (a page belongs to exactly one archetype, which maps cleanly to a review workflow), and it lets us pick k transparently via silhouette rather than relying on a density parameter that's harder to justify to a non-technical reviewer. Alternative methods remain a reasonable future extension (noted as a stretch goal from the recent review), not a requirement for this baseline model.

*   Fits the feature frame directly — the 5-feature frame from w03 (gsc_impressions, gsc_avg_position, gsc_clicks, content_age_days, word_count), all numeric after preprocessing (log-transform, imputation, missingness flags, scaling), is exactly the kind of continuous feature space K-Means is built for — a distance-based method needs numeric, scaled inputs, which this frame provides after the preprocessing work already done and verified.

What this method can and cannot do, stated up front: K-Means will produce a fixed number of behavioral groups based on distance in the scaled feature space — it does not predict, does not rank by priority, and does not explain why a page behaves a certain way. It answers "what recurring patterns exist," consistent with the lane's original research question from w01, not "what should be done about it" — that interpretive step happens afterward, by inspecting real cluster profiles and examples, not by the algorithm itself.





## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Per the lane guide's Validation Rules section — "client/group holdout, when pages from the same client may share patterns the model could memorize" — this project uses a client-grouped split: GroupShuffleSplit on client_hash_id, 75/25, producing 35 train clients / 133,474 rows and 12 validation clients / 43,264 rows, with zero client overlap between the two sets (verified directly in output, not assumed).

Why grouped-by-client, over the toolkit's other split options:

*   Not a plain random row split — pages from the same client are not independent observations. A single client's content likely shares a CMS template, an editorial word-count convention, and a consistent GSC tracking setup, so pages from the same client will naturally sit close together in feature space regardless of any real archetype. A random split would let near-duplicate client patterns leak into both train and validation, inflating the silhouette and Davies-Bouldin scores by rewarding the model for memorizing a client's "house style" rather than discovering a genuine cross-client behavioral pattern. This is the exact failure mode the leakage checklist warns against: "are duplicate or related rows split across train and test in a way that makes the test too easy?"

*  Not a time-aware split — a time-aware train/test split is the right design when the question is about predicting a future outcome from a prior window (the lane guide reserves this explicitly for the Growth/Recovery/Momentum freestyle direction). This notebook's population is a single fixed window — March 2026, gsc_data_available=TRUE — aggregated to one row per (client, content) with no forward-looking target. Lane 3's question, established in w01/w02, is "what recurring behavioral archetypes exist," not "what will happen next" — there is no future window to hold out, so a time split would be solving a problem this lane doesn't have.

*  Client-grouped is what the modeling population's own structure demands — the sanity checks already run in this notebook (Cell 14's per-cluster top-client-share numbers: 14.0%–39.3%) confirm that client identity is a real, measurable source of correlation within the feature space. A validation design that ignores this would be validating against noise it already knows exists.

What this split can and cannot prove: a client-grouped holdout tells us whether the cluster structure generalizes to clients the model has never seen — which is the right bar for an archetype system meant to apply across FlyRank's whole client base, not just the training clients. It does not tell us whether the structure is stable over time for a given client (that would need a time-aware design layered on top, and is out of scope for this single-month, cross-sectional clustering question). The val-set silhouette (0.3568) and Davies-Bouldin (0.9087) reported for k=4 are therefore honestly comparable to the train-set numbers precisely because the split guarantees no client-level shortcut was available to either side.





## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.